# Animation Scalar Properties

Generate kaleidocycle animations, compute scalar diagnostics, and plot their evolution over frames.

In [ ]:
%matplotlib widget

import matplotlib.pyplot as plt
import numpy as np

from kaleidocycle import KaleidocycleAnimation, generate_animation, random_hinges


In [ ]:
def plot_scalar_property_evolution(anim: KaleidocycleAnimation, property_names: list[str]):
    fig, axes = plt.subplots(len(property_names), 1, figsize=(9, 2.8 * len(property_names)), sharex=True)
    if len(property_names) == 1:
        axes = [axes]

    frames = np.arange(anim.n_frames)
    for ax, prop_name in zip(axes, property_names):
        values = anim.scalar_properties[prop_name]
        ax.plot(frames, values, linewidth=2, marker="o", markersize=3)
        ax.set_ylabel(prop_name.replace("_", " ").title())
        ax.grid(True, alpha=0.3)

    axes[-1].set_xlabel("Frame")
    fig.tight_layout()
    return fig


## Sine-Gordon Evolution

In [ ]:
hinges = random_hinges(8, seed=42, oriented=True).as_array()
frames = generate_animation(
    hinges,
    num_frames=50,
    step_size=0.02,
    rule="sine-Gordon",
    oriented=True,
)
anim = KaleidocycleAnimation(frames=frames, evolution_rule="sine-Gordon")

for prop in ["bending_energy", "mean_torsion", "mean_curvature"]:
    anim.compute_scalar_property(prop)

print(f"frames: {anim.n_frames}, vertices per frame: {anim.n_vertices}")
print(f"computed properties: {list(anim.scalar_properties)}")
plot_scalar_property_evolution(anim, ["bending_energy", "mean_torsion", "mean_curvature"])


## Step Evolution

In [ ]:
hinges = random_hinges(6, seed=123, oriented=False).as_array()
frames = generate_animation(
    hinges,
    num_frames=30,
    step_size=0.05,
    rule="step",
    oriented=False,
    verbose=False,
)
step_anim = KaleidocycleAnimation(frames=frames, evolution_rule="step")

for prop in ["bending_energy", "mean_torsion", "mean_curvature"]:
    step_anim.compute_scalar_property(prop)

plot_scalar_property_evolution(step_anim, ["bending_energy", "mean_torsion", "mean_curvature"])


## Compare Bending Energy Across Sizes

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4.5))

for n in [6, 8, 10]:
    hinges = random_hinges(n, seed=42, oriented=True).as_array()
    frames = generate_animation(
        hinges,
        num_frames=30,
        step_size=0.02,
        rule="sine-Gordon",
        oriented=True,
    )
    comparison = KaleidocycleAnimation(frames=frames, evolution_rule="sine-Gordon")
    comparison.compute_scalar_property("bending_energy")
    ax.plot(comparison.scalar_properties["bending_energy"], marker="o", markersize=3, label=f"n={n}")

ax.set_xlabel("Frame")
ax.set_ylabel("Bending Energy")
ax.legend()
ax.grid(True, alpha=0.3)
fig.tight_layout()


## Custom Scalar Property

In [ ]:
from kaleidocycle.geometry import binormals_to_tangents, pairwise_curvature


def max_curvature(hinges: np.ndarray) -> float:
    tangents = binormals_to_tangents(hinges, normalize=True)
    curvature = pairwise_curvature(hinges, tangents)
    return float(np.max(np.abs(curvature)))

custom_anim = KaleidocycleAnimation(
    frames=generate_animation(random_hinges(8, seed=42, oriented=True).as_array(), num_frames=30, rule="sine-Gordon"),
    evolution_rule="sine-Gordon",
)
custom_anim.compute_scalar_property("mean_curvature")
custom_anim.compute_scalar_property("max_curvature", func=max_curvature)
plot_scalar_property_evolution(custom_anim, ["mean_curvature", "max_curvature"])
